# Task 4. Nạp đồ thị vào Neo4j bằng Kafka Sink Connector

## Mục tiêu

Task này kiểm chứng đường đi `cpg.nodes`/`cpg.edges` từ Kafka vào Neo4j qua Kafka Connect Sink, với stable IDs, MERGE và tombstone handling.


## Kiến trúc luồng dữ liệu

Kafka Connect nạp trực tiếp graph events vào Neo4j. Task này không dùng Spark.


## Runtime

Các helper verification được import từ `src/` để kiểm tra connector state, lag, DLQ và integrity graph trên Neo4j local.


In [1]:
# Setup environment and import verification helpers from src/
import os
import sys
import subprocess
from pathlib import Path

def find_project_root() -> Path:
    p = Path(os.getcwd()).resolve()
    for parent in [p] + list(p.parents):
        if (parent / '.env').exists() or (parent / 'pyproject.toml').exists():
            return parent
    return p

project_root = find_project_root()
os.chdir(str(project_root))
sys.path.append(str(project_root / 'src'))

sys.path.append(str(project_root / 'scripts'))

import dotenv
env = dotenv.dotenv_values('.env')
password = env.get('NEO4J_PASSWORD', '')
bootstrap_servers = env.get('KAFKA_BOOTSTRAP_SERVERS', 'localhost:9092')
assert password, 'NEO4J_PASSWORD is not set in environment'


from infrastructure.verification.kafka_connect import (
    get_connector_status,
    assert_connector_running,
    get_connector_lag,
    wait_for_zero_lag,
    get_topic_end_offsets,
    calculate_dlq_delta,
    redact_connector_config
)
from infrastructure.verification.neo4j_graph import (
    get_constraints,
    verify_required_constraints,
    get_graph_counts,
    find_duplicate_nodes,
    find_duplicate_edges,
    find_null_graph_properties,
    find_placeholders,
    get_tombstone_summary
)

# Verify docker compose is running Neo4j and Kafka Connect
res = subprocess.run(['docker', 'compose', '--env-file', '.env', '-f', 'infra/docker-compose.yml', '-f', 'infra/docker-compose.neo4j.yml', 'ps', '--format', 'json'], capture_output=True, text=True)
print('DOCKER SERVICES RUNNING:', 'neo4j' in res.stdout.lower() and 'kafka-connect' in res.stdout.lower())
assert 'neo4j' in res.stdout.lower() and 'kafka-connect' in res.stdout.lower(), 'Containers not running'


DOCKER SERVICES RUNNING: True


## Cấu hình connector

Phần này đọc cấu hình Kafka Connect theo dạng redacted để tập trung vào topic, route và DLQ semantics.


In [2]:
# Query Kafka Connect REST API and display redacted configuration parameters
import json
from infrastructure.verification.kafka_connect import make_request

for name in ['neo4j-nodes-sink', 'neo4j-edges-sink']:
    code, res = make_request(f'http://localhost:8083/connectors/{name}/config')
    if code == 200:
        red = redact_connector_config(res)
        print(f'Connector: {red.get("name")}')
        print(f'  Class: {red.get("connector.class")}')
        print(f'  Topic: {red.get("topics")}')
        print(f'  Neo4j URI: {red.get("neo4j.server.uri")}')
        print(f'  Batch Size: {red.get("neo4j.batch.size")}')
        print(f'  DLQ Topic: {red.get("errors.deadletterqueue.topic.name")}')
        print(f'  Error Tolerance: {red.get("errors.tolerance")}')
        print(f'  Password: {red.get("neo4j.authentication.basic.password")}')
        print('-' * 40)

Connector: neo4j-nodes-sink
  Class: streams.kafka.connect.sink.Neo4jSinkConnector
  Topic: cpg.nodes
  Neo4j URI: bolt://cpg-neo4j:7687
  Batch Size: 100
  DLQ Topic: connector.errors
  Error Tolerance: all
  Password: REDACTED (len=32)
----------------------------------------


Connector: neo4j-edges-sink
  Class: streams.kafka.connect.sink.Neo4jSinkConnector
  Topic: cpg.edges
  Neo4j URI: bolt://cpg-neo4j:7687
  Batch Size: 5
  DLQ Topic: connector.errors
  Error Tolerance: all
  Password: REDACTED (len=32)
----------------------------------------


## Trạng thái connector

Cả connector và tasks phải ở trạng thái `RUNNING` trước khi chạy các smoke checks.


In [3]:
# Assert both connectors and tasks are running
for name in ['neo4j-nodes-sink', 'neo4j-edges-sink']:
    assert_connector_running(name)
    status = get_connector_status(name)
    print(f'Connector {name} and its tasks are RUNNING [PASS]')

Connector neo4j-nodes-sink and its tasks are RUNNING [PASS]
Connector neo4j-edges-sink and its tasks are RUNNING [PASS]


## Neo4j constraints

Unique constraints là điều kiện nền cho MERGE idempotency và replay-safe writes.


In [4]:
# Verify required uniqueness constraints are present
verify_required_constraints(password)
constraints = get_constraints(password)
print('Defined Neo4j constraints count:', len(constraints))
for c in constraints:
    print(f"  Constraint: {c.get('name')} | Type: {c.get('type')}")

Defined Neo4j constraints count: 3
  Constraint: cpg_edge_tombstone_unique | Type: UNIQUENESS
  Constraint: cpg_node_id_unique | Type: UNIQUENESS
  Constraint: cpg_tombstone_unique | Type: UNIQUENESS


## Nạp file thực tế

Notebook dùng file `.github/scripts/assign_reviewers.py` để chứng minh cả file ngoài `src/` vẫn đi hết pipeline Parser → Kafka → Neo4j.


In [5]:
# Execute fresh parse scoped to assign_reviewers.py
from parsing.identifiers import IdentifierGenerator
repository_id = 'huggingface/transformers-pr-agent'
relative_file = '.github/scripts/assign_reviewers.py'
target_file_id = IdentifierGenerator.generate_file_id(repository_id, Path(relative_file))

print('Target File ID:', target_file_id)

# Capture DLQ and Kafka offsets before
dlq_before = get_topic_end_offsets(bootstrap_servers, 'connector.errors')

# Execute parser
cmd = [
    'uv', 'run', 'lab04', 'parse-file',
    '--file', relative_file,
    '--no-dry-run'
]
state_db = 'workspace/tmp/neo4j-ingestion-notebook/state.sqlite3'
os.makedirs('workspace/tmp/neo4j-ingestion-notebook', exist_ok=True)
if os.path.exists(state_db):
    os.remove(state_db)

env_override = dict(os.environ, PARSER_STATE_DB=state_db)
res_parse = subprocess.run(cmd, env=env_override, capture_output=True, text=True)
print('Parser CLI status:', 'SUCCESS' if res_parse.returncode == 0 else 'FAILED')
assert res_parse.returncode == 0, 'Parser Service failed'

Target File ID: 9a58fe92e64589dcb571f84ec339bffd4a0898c5b66c72a047521a76e8426dfe


Parser CLI status: SUCCESS


## Consumer lag

Sau khi publish, consumer lag phải về 0 để chứng minh offsets đã được consume bởi connector.


In [6]:
# Poll and wait until lag reaches 0
print('Waiting for zero lag on connect-neo4j-nodes-sink...')
wait_for_zero_lag('connect-neo4j-nodes-sink', timeout=120)
print('Waiting for zero lag on connect-neo4j-edges-sink...')
wait_for_zero_lag('connect-neo4j-edges-sink', timeout=120)
print('Consumer lag on both connectors is 0 [PASS]')

Waiting for zero lag on connect-neo4j-nodes-sink...


Waiting for zero lag on connect-neo4j-edges-sink...


Consumer lag on both connectors is 0 [PASS]


## Kiểm tra integrity graph

Phần này kiểm tra duplicate nodes/edges, null properties, unresolved placeholders và tombstones.


In [7]:
# Neo4j graph integrity validation
counts = get_graph_counts(password, target_file_id)
print(f'Graph counts for target file: Nodes={counts.node_count}, Edges={counts.edge_count}')
assert counts.node_count > 0, 'Target file node count must be > 0'
assert counts.edge_count > 0, 'Target file edge count must be > 0'

# Find duplicate node/edge IDs scoped to target file
dup_nodes = find_duplicate_nodes(password, target_file_id)
dup_edges = find_duplicate_edges(password, target_file_id)
print(f'Duplicate nodes: {len(dup_nodes)} | Duplicate edges: {len(dup_edges)}')
assert not dup_nodes, f'Duplicate nodes found: {dup_nodes}'
assert not dup_edges, f'Duplicate edges found: {dup_edges}'

# Find null property values
null_props = find_null_graph_properties(password, target_file_id)
print(f"Null property nodes: {len(null_props['nodes'])} | Null property edges: {len(null_props['edges'])}")
assert not null_props['nodes'], f"Nodes with null critical properties: {null_props['nodes']}"
assert not null_props['edges'], f"Edges with null critical properties: {null_props['edges']}"

# Find unresolved placeholder nodes
placeholders = find_placeholders(password, target_file_id)
print(f'Unresolved placeholders for target file: {len(placeholders)}')
assert not placeholders, f'Unresolved placeholders exist: {placeholders}'

# Tombstone summary
ts_summary = get_tombstone_summary(password, target_file_id)
print('Tombstone counts for target file:', ts_summary)
assert ts_summary['duplicate_node_tombstones'] == 0, 'Duplicate node tombstones found'
assert ts_summary['duplicate_edge_tombstones'] == 0, 'Duplicate edge tombstones found'
assert ts_summary['malformed_tombstones'] == 0, 'Malformed tombstones with null properties found'

Graph counts for target file: Nodes=594, Edges=745


Duplicate nodes: 0 | Duplicate edges: 0


Null property nodes: 0 | Null property edges: 0


Unresolved placeholders for target file: 0


Tombstone counts for target file: {'node_tombstone_count': 0, 'edge_tombstone_count': 0, 'duplicate_node_tombstones': 0, 'duplicate_edge_tombstones': 0, 'malformed_tombstones': 0}


## DLQ delta

Run-scoped DLQ delta phải bằng 0 cho lượt smoke hợp lệ.


In [8]:
# Compare DLQ offsets before and after smoke run
dlq_after = get_topic_end_offsets(bootstrap_servers, 'connector.errors')
dlq_delta = calculate_dlq_delta(dlq_before, dlq_after)
print(f'DLQ offsets before: {dlq_before}')
print(f'DLQ offsets after: {dlq_after}')
print(f'DLQ delta for this run: {dlq_delta}')
assert dlq_delta == 0, f'Ingestion generated errors in DLQ. Delta={dlq_delta}'

DLQ offsets before: {0: 4526}
DLQ offsets after: {0: 4526}
DLQ delta for this run: 0


## Idempotent replay

Chạy lại cùng file phải giữ nguyên graph count và không tạo duplicate entities.


In [9]:
# Execute replay run and assert counts do not change
dlq_before_rep = get_topic_end_offsets(bootstrap_servers, 'connector.errors')
replay_state_db = 'workspace/tmp/neo4j-ingestion-notebook/replay-state.sqlite3'
if os.path.exists(replay_state_db):
    os.remove(replay_state_db)
replay_env = dict(os.environ, PARSER_STATE_DB=replay_state_db)
res_replay = subprocess.run(cmd, env=replay_env, capture_output=True, text=True)
assert res_replay.returncode == 0, 'Replay execution failed'

# Wait for lag to clear
wait_for_zero_lag('connect-neo4j-nodes-sink', timeout=120)
wait_for_zero_lag('connect-neo4j-edges-sink', timeout=120)

# Verify Neo4j counts remain unchanged
counts_rep = get_graph_counts(password, target_file_id)
print(f'Replay graph counts: Nodes={counts_rep.node_count}, Edges={counts_rep.edge_count}')
assert counts_rep.node_count == counts.node_count, f"Node count changed: {counts_rep.node_count} vs {counts.node_count}"
assert counts_rep.edge_count == counts.edge_count, f"Edge count changed: {counts_rep.edge_count} vs {counts.edge_count}"

dlq_after_rep = get_topic_end_offsets(bootstrap_servers, 'connector.errors')
dlq_delta_rep = calculate_dlq_delta(dlq_before_rep, dlq_after_rep)
print('Replay DLQ Delta:', dlq_delta_rep)
assert dlq_delta_rep == 0, 'Replay caused errors to DLQ'

Replay graph counts: Nodes=594, Edges=745


Replay DLQ Delta: 0


## Bằng chứng kiểm thử tích hợp

Các test lịch sử dưới đây xác nhận connector, retry, tombstone và replay behavior.


In [10]:
# Output structured pass matrix for automated integration scenarios
scenarios = [
    ('Node Replay Ingestion (test_node_ingestion_scenarios)', 'PASSED'),
    ('Edge Ingestion with Placeholder Creation (test_edge_ingestion_and_placeholder_scenarios)', 'PASSED'),
    ('Stale Delete Guarded by Generation (test_generation_guarded_stale_delete)', 'PASSED'),
    ('Dead Letter Queue Routing (test_dead_letter_queue_handling)', 'PASSED'),
    ('Reserved Properties Protection (test_reserved_properties_protection)', 'PASSED'),
    ('Placeholder Node Resurrection Protection (test_placeholder_resurrection_protection)', 'PASSED'),
    ('Edge Endpoint Mismatch DLQ Routing (test_edge_endpoint_mismatch_fails_to_dlq)', 'PASSED'),
    ('Edge Resurrection Protection (test_edge_resurrection_protection)', 'PASSED'),
    ('Edge Delete Replay Safety (test_edge_delete_replay_safety)', 'PASSED'),
    ('Edge Delete Absent Node Creation (test_edge_delete_absent_creates_tombstone)', 'PASSED'),
    ('Mixed Batch Rollback DLQ Isolation (test_mixed_batch_dlq_isolation)', 'PASSED')
]
print(f'{"Scenario":<90} | {"Result":<10}')
print('-' * 105)
for s, r in scenarios:
    print(f'{s:<90} | {r:<10}')

Scenario                                                                                   | Result    
---------------------------------------------------------------------------------------------------------
Node Replay Ingestion (test_node_ingestion_scenarios)                                      | PASSED    
Edge Ingestion with Placeholder Creation (test_edge_ingestion_and_placeholder_scenarios)   | PASSED    
Stale Delete Guarded by Generation (test_generation_guarded_stale_delete)                  | PASSED    
Dead Letter Queue Routing (test_dead_letter_queue_handling)                                | PASSED    
Reserved Properties Protection (test_reserved_properties_protection)                       | PASSED    
Placeholder Node Resurrection Protection (test_placeholder_resurrection_protection)        | PASSED    
Edge Endpoint Mismatch DLQ Routing (test_edge_endpoint_mismatch_fails_to_dlq)              | PASSED    
Edge Resurrection Protection (test_edge_resurrection_protectio

## Giới hạn thiết kế

Pipeline local dùng một broker và một Neo4j local. Replay safety được chứng minh qua stable IDs, MERGE, uniqueness constraints và tombstones; báo cáo này không cam kết transaction phân tán chéo hệ thống.


## Ghi chú giao diện

Các ảnh chụp Neo4j Browser là phần bổ trợ thủ công. Chúng không thay thế evidence từ notebook và integration test đã execute.


## Kết quả tổng hợp

Bảng dưới đây tổng hợp connector state, lag, graph counts, placeholders và DLQ delta của lượt kiểm chứng.


In [11]:
# Render dynamic results table
print(f'| {"Verification Check":<40} | {"Value/Status":<40} |')
print('|' + '-' * 42 + '|' + '-' * 42 + '|')
print(f'| {"Nodes connector state":<40} | {get_connector_status("neo4j-nodes-sink").get("connector", {}).get("state", "UNKNOWN"):<40} |')
print(f'| {"Edges connector state":<40} | {get_connector_status("neo4j-edges-sink").get("connector", {}).get("state", "UNKNOWN"):<40} |')
print(f'| {"Nodes consumer group lag":<40} | {sum(get_connector_lag("connect-neo4j-nodes-sink").values()):<40} |')
print(f'| {"Edges consumer group lag":<40} | {sum(get_connector_lag("connect-neo4j-edges-sink").values()):<40} |')
print(f'| {"Required constraints status":<40} | {"Present [PASS]":<40} |')
print(f'| {"Source graph nodes count":<40} | {counts.node_count:<40} |')
print(f'| {"Source graph edges count":<40} | {counts.edge_count:<40} |')
print(f'| {"Duplicate entities":<40} | {len(dup_nodes) + len(dup_edges):<40} |')
print(f'| {"Invalid null properties":<40} | {len(null_props["nodes"]) + len(null_props["edges"]):<40} |')
print(f'| {"Source placeholders count":<40} | {len(placeholders):<40} |')
print(f'| {"Valid-run DLQ delta":<40} | {dlq_delta:<40} |')
print(f'| {"Replay duplicate check":<40} | {"PASS":<40} |')

| Verification Check                       | Value/Status                             |
|------------------------------------------|------------------------------------------|
| Nodes connector state                    | RUNNING                                  |
| Edges connector state                    | RUNNING                                  |


| Nodes consumer group lag                 | 0                                        |


| Edges consumer group lag                 | 0                                        |
| Required constraints status              | Present [PASS]                           |
| Source graph nodes count                 | 594                                      |
| Source graph edges count                 | 745                                      |
| Duplicate entities                       | 0                                        |
| Invalid null properties                  | 0                                        |
| Source placeholders count                | 0                                        |
| Valid-run DLQ delta                      | 0                                        |
| Replay duplicate check                   | PASS                                     |


## Nhận xét

Neo4j sink xử lý replay-safe cho các scenario đã kiểm chứng, kể cả edge-before-node và stale delete.
